# Training and evaluating a laser-melting surrogate, with and without AiiDA

This notebook trains and evaluates the neural-operator surrogates of this repository (FNO, RIGNO and GAOT) on a tiny
synthetic dataset, along two routes:

* **Route A, without AiiDA.** The Hydra entry points of `lpno` run directly on folders of NPZ files.
  Nothing is recorded beyond the run folders. Use it to debug, to try a setting quickly, or on a machine without an
  AiiDA profile, such as a cluster job (`reproducibility/rerun/README.md` replays the published runs this way).
* **Route B, with AiiDA.** The `aiida-pino` plugin imports the simulations as nodes, composes dataset collections and
  runs training and evaluation as AiiDA processes. Every model and every evaluation becomes a node linked to the data
  and the settings that produced it. The published models and evaluations were made this way.

In both routes the architecture is chosen by the Hydra config (`--config-name`; each config sets
`custom.model_family`) and sized by the `arch.fno.*`, `arch.rigno.*` or `arch.gaot.*` keys. The training
hyperparameters are ordinary Hydra keys (`training.max_steps`, `scheduler.max_lr`, `batch_size.grid`, ...). Route B
passes the same keys through a training profile.

**The data are synthetic.** `synthetic_data.py` (beside this notebook) generates twelve straight-track
"simulations" from an analytic moving-heat-source solution. They carry the fields and metadata the loaders expect and
are small enough for a CPU, so they exercise the pipeline and nothing else: a model trained on them says nothing about
laser melting. The FLOW-3D WELD output behind the paper is in the restricted-access record of the data deposit
(doi:10.5281/zenodo.23325088, shared on request for non-commercial use). The trained models, the evaluations and the
metadata of every simulation are in the open record (doi:10.5281/zenodo.23325086). Section 5 re-runs one published
evaluation from them when both are available, and is skipped otherwise.

**What you need**

* The environment of `tools/setup_env.py` with AiiDA (`python3.10 tools/setup_env.py --prefix <env> --torch cpu
  --with-aiida`), and a Jupyter kernel running its Python. It provides `lpno`, `aiida-pino`,
  `nvidia-physicsnemo-sym` and `torch-scatter` (optional: RIGNO and GAOT fall back to native torch scatter ops
  without it).
* No GPU and no message broker. The notebook sets `PINO_ALLOW_CPU=1` for every training and evaluation, and runs the
  AiiDA processes in the foreground.
* About 10 minutes of CPU time at 4 threads per process (measured on a laptop; 8 cluster cores should not be slower),
  and about 200 MB of disk.

Run it unattended from `reproducibility/`:

```bash
cd reproducibility
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=1800 \
    train_evaluate.ipynb --output train_evaluate.executed.ipynb
```

Everything is written under one work folder, `reproducibility/train_evaluate_work/` (or `$LPNO_TUTORIAL_WORK`), which
is deleted and rebuilt on every run. The AiiDA profile of route B lives there too, so `~/.aiida` is not touched.

## 0. Setup

Paths, and the environment every subprocess receives. One setting is specific to a machine without a GPU:
`PINO_ALLOW_CPU=1`. The entry points stop at start-up when CUDA is unavailable unless it is set (`configure_device`
in `lpno/models/pipeline/config_loading.py`). With it they run on the CPU; on a CPU-only build of torch
they also turn physicsnemo's NVTX profiler calls, which such a build does not have, into no-ops
(`models/pipeline/nvtx_guard.py`).

In [ ]:
import configparser
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

HERE = Path.cwd()                                    # reproducibility/
CODE = Path(os.environ.get("LPNO_CODE", HERE.parent)).resolve()   # the code repository
sys.path[:0] = [str(HERE), str(CODE / "reproducibility")]
import synthetic_data as sd                          # the synthetic cases (beside this notebook)
import lpno_repro as lr                              # the metric conventions of the paper's tables

WORK = Path(os.environ.get("LPNO_TUTORIAL_WORK", HERE / "train_evaluate_work")).resolve()
if WORK.exists():                                    # start from scratch, but only in a folder this notebook made
    if not (WORK / ".lpno_tutorial").exists():
        raise SystemExit(f"{WORK} exists and was not made by this notebook; set LPNO_TUTORIAL_WORK elsewhere")
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
(WORK / ".lpno_tutorial").touch()

BIN = Path(sys.executable).parent                    # verdi and aiida-pino of this environment
VERDI, AIIDA_PINO = BIN / "verdi", BIN / "aiida-pino"
PROFILE = "lpno-tutorial"                            # the AiiDA profile route B creates in WORK
HAS_SCATTER = importlib.util.find_spec("torch_scatter") is not None   # optional since 2026-10-29
print("python  ", sys.executable)
print("torch   ", torch.__version__, "(CUDA build)" if torch.version.cuda else "(CPU-only build)")
print("torch_scatter", "found" if HAS_SCATTER else "missing: RIGNO and GAOT use native torch scatter ops")
print("code    ", CODE)
print("work    ", WORK)

In [ ]:
try:
    n_cores = len(os.sched_getaffinity(0))           # what Slurm or taskset gave this process
except AttributeError:
    n_cores = os.cpu_count() or 1
THREADS = os.environ.get("OMP_NUM_THREADS", str(min(8, n_cores)))

ENV = {**os.environ,
       "PINO_ALLOW_CPU": "1",                        # let the entry points run without CUDA
       "WANDB_MODE": "offline",                      # no Weights & Biases upload
       "OMP_NUM_THREADS": THREADS, "MKL_NUM_THREADS": THREADS,
       "PINO_LOADER_WORKERS": "0",                   # load batches in the training process itself
       "MPLBACKEND": "Agg",
       "AIIDA_PATH": str(WORK / "aiida"),            # route B's AiiDA profile lives in WORK
       "AIIDA_PINO_RUNTIME_CACHE": str(WORK / "aiida_runtime_cache"),
       "AIIDA_PINO_EXPORT_WORKERS": "1",
       "PINO_CONDUCTION_CACHE": str(WORK / "conduction_cache")}

# The same settings for jobs that AiiDA launches (route B): the prepend text of the AiiDA code.
prepend = ["export PINO_ALLOW_CPU=1", "export WANDB_MODE=offline", f"export OMP_NUM_THREADS={THREADS}"]
print(f"{THREADS} threads per process")

In [ ]:
def run(cmd, log, env=ENV, cwd=WORK, show=None, tail=8):
    # Run a command with its output in WORK/logs/<log>.txt. Print the lines matching `show`
    # (a regex) or else the last `tail` lines; raise if it fails.
    path = WORK / "logs" / f"{log}.txt"
    path.parent.mkdir(exist_ok=True)
    t0 = time.time()
    with open(path, "w") as f:
        rc = subprocess.call([str(c) for c in cmd], stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd)
    text = path.read_text()
    lines = text.splitlines()
    picked = [l for l in lines if re.search(show, l)] if show else lines[-tail:] if tail else []
    print("\n".join(l[:300] for l in picked))
    print(f"[{log}: exit {rc} after {time.time() - t0:.0f} s; output in {path.relative_to(WORK)}]")
    if rc:
        print("\n".join(lines[-30:]))
        raise RuntimeError(f"{log} failed (exit {rc}); see {path}")
    return text


def last_json(text):
    # The JSON document that ends a command's output (log lines may come before it).
    starts = [0] + [i + 1 for i, c in enumerate(text) if c == "\n"]
    for i in starts:
        if text[i:i + 1] in ("{", "["):
            try:
                return json.loads(text[i:])
            except json.JSONDecodeError:
                pass
    raise ValueError("no JSON document at the end of the output")


# Load AiiDA nodes by uuid in a separate process, so this kernel never loads a profile itself.
DESCRIBE = r'''
import json, sys
from aiida import load_profile, orm
load_profile(sys.argv[1] or None)
out = {}
for ref in sys.argv[2:]:
    node = orm.load_node(ref)
    rec = {"uuid": node.uuid, "pk": node.pk, "label": node.label, "type": type(node).__name__}
    if isinstance(node, orm.ProcessNode):
        rec["exit_status"] = node.exit_status
        rec["outputs"] = {k: node.outputs[k].uuid for k in node.outputs}
        if "training_profile" in node.inputs:
            rec["training_profile"] = node.inputs.training_profile.get_dict()
    if "evaluation_metadata" in node.base.attributes.keys():
        rec["evaluation_metadata"] = node.base.attributes.get("evaluation_metadata")
    out[ref] = rec
print(json.dumps(out, default=str))
'''


def describe(refs, profile=PROFILE, env=ENV):
    return last_json(run([sys.executable, "-c", DESCRIBE, profile or "", *refs], "describe", env=env, tail=0))


SHOW_AIIDA = r"REPORT.*(Submitted|substitution|Materialising|refus|fail)"

## 1. A tiny synthetic dataset

`synthetic_data.py` writes twelve straight-track cases on a 20 µm grid, with laser power from 120 to 320 W and scan
speed from 0.3 to 1.0 m/s, split 8 train / 2 validation / 2 test. The temperature of each case is a regularised
Rosenthal moving-point-source field (Ti-6Al-4V-like constants, absorptivity 0.35), capped smoothly a little above
boiling. The metal volume fraction is 1 below the surface at z = 0 and 0 in the ten gas cells above it, with a shallow
dimple under the beam that deepens with P/√V. Nothing flows.

The cases are written twice, once per route:

* `steady/{train,val,test}/<case>.npz`, each with a `<case>.manifest.json` beside it: one beam-centred frame per case
  on a 40 × 16 × 22 grid, with the beam at x = 0 (column 27). This is the layout the steady models of the architecture
  table were trained on (the `config_flow3D_*_pvlhs_steady` configs). The files are in the consolidated-NPZ format that
  `lpno/src/lpno/custom_utils/npz_datasource.py` reads: axes `x`, `y`, `z` (m) and `time_s`
  (s), the scalars `power_w`, `scan_speed_m_per_s` and `case_name`, and fields shaped `(nt, nx, ny, nz)` under their
  FLOW-3D names, here `Temperature` (K) and `Fraction Of Fluid`.
* `runs/<case>/case_manifest.json` and `runs/<case>/npz/<case>.npz`: the same cases as eight laboratory-frame frames,
  the beam moving one cell per frame, in the run-directory layout that `flow3D_scripts` writes and aiida-pino imports.
  aiida-pino derives the steady layout from these itself in route B; its five-frame beam-following average equals
  the steady files to within 0.01 K.

In [ ]:
DATA = WORK / "data"
STEADY = DATA / "steady"
sd.write_steady_split(STEADY)
sd.write_flow3d_runs(DATA / "runs", n_frames=8, steady_frames=5)

CASES = sd.default_cases()
SPLIT = {c.name: c.split for c in CASES}
pd.DataFrame([{"case": c.name, "split": c.split, **sd.summary(STEADY / c.split / f"{c.name}.npz")}
              for c in CASES]).round(2)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

case = STEADY / "train" / "synth_000.npz"
with np.load(case) as z:
    print({k: z[k].shape for k in z.files})
    x, zz = z["x"] * 1e6, z["z"] * 1e6
    T, metal = z["Temperature"][0], z["Fraction Of Fluid"][0]
manifest = json.loads(case.with_suffix(".manifest.json").read_text())
print("manifest:", sorted(manifest), "| laser:", manifest["input"]["laser"])

j = T.shape[1] // 2                                   # the section next to the track centre line
fig, axes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
for ax, field, label in ((axes[0], T[:, j].T, "Temperature (K)"), (axes[1], metal[:, j].T, "Metal fraction")):
    fig.colorbar(ax.pcolormesh(x, zz, field, shading="nearest"), ax=ax, label=label)
    ax.set_xlabel("x relative to the beam (µm)")
axes[0].contour(x, zz, T[:, j].T, [sd.T_LIQUIDUS_K], colors="white")
axes[0].set_ylabel("z (µm)")
plt.show()

## 2. Route A: `lpno` without AiiDA

Training is `python -m lpno.models.NS_Energy_laser_PINO --config-name=<config> key=value ...`, and
evaluation is `python -m lpno.models.NS_Energy_laser_PINO_eval --config-name=<config>
network_dir=<trained run> custom.data_dir=<split> ...`. The configs are in
`lpno/src/lpno/conf/hydra/`. The three steady configs of the architecture table share the
input keys `[V_scan, P, H_norm, x, y, z]` (the pipeline adds `t_hat`), the outputs `[T, alpha]`, the one-cycle
schedule and the 32,000-step budget, so they differ only in the architecture:

| architecture | config | size keys (published value → here) |
| --- | --- | --- |
| FNO | `config_flow3D_FNO_pvlhs_steady` | `arch.fno.fno_modes` [20,18,8] → [8,6,6], `arch.fno.nr_fno_layers` 3 → 2, `arch.fno.padding` 9 → 4; the width is the latent size in `arch.decoder.input_keys=[z,<width>]`, 32 → 16; decoder MLP `arch.decoder.layer_size` 90 → 32, `nr_layers` 3 → 2 |
| RIGNO | `config_flow3D_RIGNO_pvlhs_steady` | `arch.rigno.node_latent_size` and `edge_latent_size` 96 → 32, `processor_steps` 18 → 2, `num_graph_variants` 8 → 1, `grad_checkpointing` true → false; also `subsample_factor`, `k_p2r`, `k_r2p`, `rmesh_levels`, `film_keys` |
| GAOT | `config_flow3D_GAOT_pvlhs_steady` | `arch.gaot.latent_tokens` [44,20,24] → [20,8,12] (every axis divisible by `patch_size` 2), `hidden_size` 256 → 64, `ffn_hidden_size` 1024 → 128, `num_layers` 5 → 2, `num_heads` 8 → 4, `lifting_channels` 32 → 16, `projection_channels` 256 → 64 |

`lpno/README.md` ("Architectures") documents the main `arch.rigno.*` and `arch.gaot.*` keys. FNO modes cannot exceed half the
cells on their axis (20, 8 and 11 here); the pipeline prints the limit at start-up.

Overrides every run below uses, and why:

* `training.max_steps`, with the recording intervals no longer than it. The pipeline refuses a run whose
  `rec_results_freq` or `rec_validation_freq` exceeds `max_steps`, or whose `stop_criterion.freq` is not a multiple of
  `rec_validation_freq` (`validate_recording_cadence` in `config_loading.py`). The weights are written every
  `training.save_network_freq` steps (physicsnemo's default, 1000), which the pipeline caps to `max_steps`, so a
  200-step run saves the weights of step 200.
* `scheduler.max_lr`, the learning rate. These configs use a one-cycle schedule, which sets the rate of every step from
  `scheduler.max_lr`; `optimizer.lr` is overwritten.
* `custom.moving_ref_frame=true custom.moving_frame_steps=1`. Each steady file holds one frame that already follows the
  beam. Saying so places the beam box (the substrate within two beam radii of the beam) at the origin, so the beam-box
  error is computed and drives checkpoint selection and the stop criterion (`beambox_mae_T`), as for the published
  steady models. With the config's own `moving_ref_frame=false` that metric cannot be computed, and the run says so and
  stops on the whole-domain `mae_T` instead.
* `network_dir` and `hydra.run.dir`, both a fresh folder per run. physicsnemo resumes from any checkpoint it finds in
  `network_dir`, so a reused folder continues the old run. The evaluation reads the run's settings back from
  `network_dir/.hydra/config.yaml`; only the data paths need repeating.

The evaluation writes `fno_eval/model_errors.txt` (for every architecture; the name is historical), with one section
per output. Temperature errors there are normalised by `ref_scales.T` (3000 K). The headline numbers below follow the
paper's conventions (`lpno_repro.py`): T MAE and RMSE over substrate cells, the beam-box MAE, the relative error over
the substrate, and the melt-pool IoU (liquid fraction above 0.5).

In [ ]:
CONFIGS = {"fno": "config_flow3D_FNO_pvlhs_steady",
           "rigno": "config_flow3D_RIGNO_pvlhs_steady",
           "gaot": "config_flow3D_GAOT_pvlhs_steady"}
RUNS_A = WORK / "route_a"
SHOW_TRAIN = r"parameters:|validation: |saved checkpoint|reached maximum"


def schedule(steps, every):
    # the step budget, with every recording interval no longer than it
    return [f"training.max_steps={steps}", f"training.rec_results_freq={every}",
            f"training.rec_validation_freq={every}", f"training.summary_freq={every}",
            f"stop_criterion.freq={every}",
            f"stop_criterion.patience={steps}", "training.print_stats_freq=10"]


COMMON = ["batch_size.grid=2", "batch_size.validation=2", "custom.inference_batch_size=2",
          "custom.moving_ref_frame=true", "custom.moving_frame_steps=1", "custom.time_avg_window=1"]


def train_a(family, name, overrides):
    out = RUNS_A / name
    run([sys.executable, "-m", "lpno.models.NS_Energy_laser_PINO", f"--config-name={CONFIGS[family]}",
         f"custom.name={name}", f"custom.data_dir={STEADY / 'train'}", f"custom.val_data_dir={STEADY / 'val'}",
         f"network_dir={out}", f"hydra.run.dir={out}", *COMMON, *overrides],
        f"train_{name}", show=SHOW_TRAIN)
    return out


def read_model_errors(path):
    # fno_eval/model_errors.txt: a [section] per output (T, alpha, fl, geometry) of `key: value` lines
    cp = configparser.ConfigParser()
    cp.optionxform = str                              # keep the keys' case
    cp.read(path)
    def number(v):
        try:
            return float(v)
        except ValueError:
            return v
    return {s: {k: number(v) for k, v in cp[s].items()} for s in cp.sections()}


def evaluate_a(family, model_dir, split="test"):
    out = RUNS_A / f"{model_dir.name}_on_{split}"
    run([sys.executable, "-m", "lpno.models.NS_Energy_laser_PINO_eval",
         f"--config-name={CONFIGS[family]}", f"custom.name={out.name}", f"network_dir={model_dir}",
         f"hydra.run.dir={out}", f"custom.data_dir={STEADY / split}", f"custom.val_data_dir={STEADY / split}"],
        f"eval_{out.name}", tail=0)
    return read_model_errors(out / "fno_eval" / "model_errors.txt")


def headline(m, t_scale=3000.0):
    T = m["T"]
    return {"T MAE (K)": T["substrate_abs_err_mean"] * t_scale, "T RMSE (K)": T["substrate_rmse"] * t_scale,
            "BB MAE (K)": T["beambox_abs_err_mean"] * t_scale, "Rel. err. (%)": 100 * T["substrate_rel_err_mean"],
            "Melt-pool IoU": m["fl"]["iou_mean"]}

### FNO

200 steps at a peak learning rate of 1e-3, batch size 2; about half a minute on 8 cores.

In [ ]:
FNO_ARCH = ["arch.fno.fno_modes=[8,6,6]",            # Fourier modes kept along x, y, z
            "arch.fno.nr_fno_layers=2",               # spectral layers
            "arch.fno.padding=4",
            "arch.decoder.input_keys=[z,16]",         # the FNO width: 16 latent channels
            "arch.decoder.layer_size=32", "arch.decoder.nr_layers=2"]   # pointwise decoder
fno_dir = train_a("fno", "fno_synth", FNO_ARCH + schedule(200, 50) + ["scheduler.max_lr=1e-3"])
print(sorted(p.name for p in fno_dir.iterdir()))

In [ ]:
results_a = {"FNO": evaluate_a("fno", fno_dir)}
print(json.dumps(results_a["FNO"]["T"], indent=1)[:900])
pd.DataFrame({k: headline(v) for k, v in results_a.items()}).T.round(3)

The run folder holds the last weights (`u_network.0.pth`), the best ones by the validation metric
(`u_network.best.pth`, recorded in `best_checkpoint.json`), the optimizer state, the training curve
(`validators/validation_metrics.csv`) and the resolved configuration (`.hydra/`). The evaluation scores the last
weights and says so; `++custom.use_best_checkpoint=true` on the evaluation command scores the best ones instead.

### RIGNO and GAOT

The same commands with another config and its own `arch` block, 200 steps each at a higher peak learning rate
(3e-3 for RIGNO, 2e-3 for GAOT); about two minutes each on 8 cores.

In [ ]:
RIGNO_ARCH = ["arch.rigno.node_latent_size=32", "arch.rigno.edge_latent_size=32",   # latent width
              "arch.rigno.processor_steps=2",          # message-passing steps on the regional graph
              "arch.rigno.num_graph_variants=1",       # regional graphs rotated during training
              "arch.rigno.grad_checkpointing=false",   # recompute in backward only to save memory
              f"arch.rigno.cache_dir={RUNS_A / 'rigno_graphs'}"]
rigno_dir = train_a("rigno", "rigno_synth", RIGNO_ARCH + schedule(200, 50) + ["scheduler.max_lr=3e-3"])
results_a["RIGNO"] = evaluate_a("rigno", rigno_dir)

In [ ]:
GAOT_ARCH = ["arch.gaot.latent_tokens=[20,8,12]",   # latent lattice, each axis divisible by patch_size
             "arch.gaot.patch_size=2",
             "arch.gaot.hidden_size=64", "arch.gaot.ffn_hidden_size=128",   # transformer width
             "arch.gaot.num_layers=2", "arch.gaot.num_heads=4", "arch.gaot.num_kv_heads=4",
             "arch.gaot.lifting_channels=16", "arch.gaot.projection_channels=64",
             f"arch.gaot.cache_dir={RUNS_A / 'gaot_graphs'}"]
gaot_dir = train_a("gaot", "gaot_synth", GAOT_ARCH + schedule(200, 50) + ["scheduler.max_lr=2e-3"])
results_a["GAOT"] = evaluate_a("gaot", gaot_dir)
pd.DataFrame({k: headline(v) for k, v in results_a.items()}).T.round(3)

These numbers measure 200 steps on eight synthetic cases. They show that each architecture trains and evaluates;
they do not rank the architectures. None of the three is converged: a model that has not yet learned where the metal
ends predicts gas there, its temperature is then replaced by the gas fill value, and a model that has not yet
produced a liquid region scores a melt-pool IoU of 0. RIGNO at this size learns the metal-gas interface slowest. The
published comparison uses 32,000 steps, matched schedules and three seeds (`tab:arch-steady`).

Other hyperparameters are set the same way: `scheduler.pct_start` and `scheduler.div_factor` shape the one-cycle
schedule, `optimizer.weight_decay` regularises, `custom.seed` sets the seed, and any `arch.*` key of the config can be
overridden. `python -m lpno.models.NS_Energy_laser_PINO --config-name=<config> --cfg job` prints the
composed configuration without training.

## 3. Route B: the same with AiiDA

aiida-pino records the steps above as a provenance graph: simulation nodes → dataset collections → a training process
→ a model node → an evaluation process → an evaluation node. Settings travel as *profiles*, dictionaries checked
against fixed key sets (`aiida-pino profile-keys <kind>` lists every key with its default):

* a **dataset profile** for the import (fields, frames, annotations);
* a **derivation profile** for the layout a job consumes (frame, time average, stride, gas crop);
* a **training profile** for the run: the config name, structured hyperparameters and a list of Hydra overrides;
* an **evaluation profile** for the scoring.

The profile below has no broker, so every submission runs in this process with `aiida-pino --foreground`.

### 3.1 Profile and code

`verdi presto` makes a SQLite profile and registers this machine as the computer `localhost`. The code is the Python
of this kernel. Its prepend text, which runs before every job, sets `PINO_ALLOW_CPU`, the offline Weights & Biases
mode and the thread count.

In [ ]:
run([VERDI, "presto", "--non-interactive", "--profile-name", PROFILE, "--email", "tutorial@localhost"], "verdi_presto")
run([VERDI, "code", "create", "core.code.installed", "--non-interactive", "--label", "pino-python",
     "--computer", "localhost", "--filepath-executable", sys.executable, "--prepend-text", "\n".join(prepend)],
    "verdi_code")
CODE_LABEL = "pino-python@localhost"
_ = run([VERDI, "status"], "verdi_status")

### 3.2 Import the simulations and compose the splits

`submit raw-flow3d` imports every run under a folder as one `PinoSimulationData` node, keeping the NPZ as it is, and
wraps them in a base collection. The split of each simulation is an annotation on its node (`aiida-pino split`), and
training wants collections that hold one split each, so the cell then composes a train, a validation and a test
collection.

In [ ]:
out = run([AIIDA_PINO, "--json", "--foreground", "submit", "raw-flow3d", "--source-path", DATA / "runs",
           "--python-code", CODE_LABEL, "--dataset-profile", "batch_keyword=synth_tutorial",
           "--dataset-profile", "description=Synthetic Rosenthal runs of train_evaluate.ipynb, not FLOW-3D output"],
          "import_runs", show=r"REPORT")
sims = last_json(run([AIIDA_PINO, "--json", "sims", "--limit", "0"], "sims", tail=0))
for s in sims:
    run([AIIDA_PINO, "split", s["pk"], SPLIT[s["case_name"]], "--reason", "tutorial split"],
        f"split_{s['case_name']}", tail=0)

COLL = {}
for split in ("train", "val", "test"):
    pks = sorted(s["pk"] for s in sims if SPLIT[s["case_name"]] == split)
    res = last_json(run([AIIDA_PINO, "--json", "collection", "create", f"synth_tutorial_{split}", *pks,
                         "--description", f"Synthetic tutorial runs, {split} split", "--keyword", "smoke"],
                        f"compose_{split}", tail=0))
    COLL[split] = res["uuid"]
run([AIIDA_PINO, "datasets"], "datasets", tail=10)
COLL

### 3.3 Training profiles

The imported runs are laboratory-frame time series. A steady training states *steady* twice, in the training profile
and in an explicit derivation profile, and the two must agree (`aiida-pino/README.md`, "Steady runs"). aiida-pino then
derives a beam-centred, five-frame-averaged collection from each split once, stores it as a new collection
(`dataset_role: derived`), and trains on that. `time_as_batch=false` has to be in both: left out of the derivation
profile it defaults to true, and the submission is refused as an incompatible variant.

The structured keys of the training profile become Hydra keys in the training command (`STRUCTURED_TO_HYDRA` in
`aiida_pino/utils/profiles.py`): `max_steps` → `training.max_steps`, `learning_rate` → `optimizer.lr` and, on a
one-cycle config, `scheduler.max_lr`, `batch_size_grid` → `batch_size.grid`, `batch_size_validation` →
`batch_size.validation`, `inference_batch_size` → `custom.inference_batch_size`, `weight_decay` →
`optimizer.weight_decay`. Everything else goes in `hydra_overrides`, applied after them, so the architecture is chosen
by `config_name` and sized there, exactly as in route A. A structured key left out is still emitted, with the config's
own value as composed with the configs it inherits (`aiida-pino profile-keys training --config <config>` lists them:
32,000 steps and batch 1 for these steady configs). The layout keys (`time_as_batch`, `moving_ref_frame`, ...) are the
exception: they default to the steady layout and are stated here anyway, as the derivation must match them.

The adapter lowers every interval longer than `max_steps` (`rec_results_freq`, `rec_validation_freq`,
`stop_criterion.freq` and the weights interval `save_network_freq`) to `max_steps`, so a short run needs no cadence
overrides and still saves its final weights. `gpu_vram_mb` declares the memory a run needs; without it
aiida-pino first runs a 100-step measuring run, which on a CPU measures nothing. The GAOT lattice has six z-tokens here
because the derivation crops the ten gas cells by default (`crop_z_top: 10`), which leaves 12 of the 22.

In [ ]:
B = WORK / "route_b"
B.mkdir()
DERIVATION = {"include_time": True, "time_as_batch": False, "moving_ref_frame": True,
              "moving_frame_steps": 5, "time_avg_window": 5}
(B / "derivation.json").write_text(json.dumps(DERIVATION))

ARCH_B = {"fno": FNO_ARCH,
          "rigno": RIGNO_ARCH[:-1],                  # graphs cached in the job folder
          "gaot": [o.replace("[20,8,12]", "[20,8,6]") for o in GAOT_ARCH[:-1]]}


BUDGET_B = {"fno": (200, 1e-3), "gaot": (100, 2e-3), "rigno": (100, 3e-3)}    # steps, peak learning rate


def training_profile(family, run_name, steps, lr):
    return {"config_name": CONFIGS[family], "run_name": run_name,
            "cuda_device": "auto", "wandb_offline": True,        # required keys
            "max_steps": steps, "learning_rate": lr,
            "batch_size_grid": 2, "batch_size_validation": 2, "inference_batch_size": 2,
            "include_time": True, "time_as_batch": False, "moving_ref_frame": True,   # steady, first time
            "moving_frame_steps": 5, "time_avg_window": 5,
            "gpu_vram_mb": 1000,
            "hydra_overrides": ARCH_B[family]}


FAMILIES_B = ["fno", "gaot", "rigno"]
for family in FAMILIES_B:
    profile = training_profile(family, f"{family}_synth_aiida", *BUDGET_B[family])
    (B / f"{family}_profile.json").write_text(json.dumps(profile, indent=1))
print((B / "fno_profile.json").read_text())

`presubmit` resolves everything a submission would (collections, derivation, staging) and reports every refusal,
without submitting anything.

In [ ]:
res = last_json(run([AIIDA_PINO, "--json", "presubmit", "--train-dataset", COLL["train"], "--val-dataset", COLL["val"],
                     "--python-code", CODE_LABEL, "--training-profile", f"@{B / 'fno_profile.json'}",
                     "--derivation-profile", f"@{B / 'derivation.json'}"], "presubmit_fno", tail=0))
print("ok:", res["ok"], "| problems:", res["problems"])
assert res["ok"], res["problems"]

### 3.4 Train

`submit train-eval --mode train` trains only (`train_eval`, the default, also scores the validation split). One
submission per architecture: FNO for 200 steps as in route A, GAOT and RIGNO for 100 to keep the notebook short. The
first submission also builds the derived collections, which the others reuse.

In [ ]:
TRAINED = {}                                          # family -> (training workflow uuid, model uuid)
for family in FAMILIES_B:
    res = last_json(run([AIIDA_PINO, "--json", "--foreground", "submit", "train-eval", "--mode", "train",
                         "--train-dataset", COLL["train"], "--val-dataset", COLL["val"], "--python-code", CODE_LABEL,
                         "--training-profile", f"@{B / f'{family}_profile.json'}",
                         "--derivation-profile", f"@{B / 'derivation.json'}",
                         "--description", f"train_evaluate.ipynb: {family} on synthetic data"],
                        f"train_{family}_aiida", show=SHOW_AIIDA))
    wf = describe([res["uuid"]])[res["uuid"]]
    assert wf["exit_status"] == 0 and "model" in wf["outputs"], wf
    TRAINED[family] = (res["uuid"], wf["outputs"]["model"])
TRAINED

### 3.5 Change a hyperparameter of a recorded run

`aiida-pino resubmit` clones a recorded training (its profiles and datasets) and re-runs it, with `--override key=value`
or `--override @file.json` applied to the training profile (`derivation.<key>` reaches the derivation profile). An
override replaces the value: a new `hydra_overrides` replaces the whole list. Here the FNO run again with a three times
lower peak learning rate.

In [ ]:
res = last_json(run([AIIDA_PINO, "--json", "--foreground", "resubmit", TRAINED["fno"][0],
                     "--override", "learning_rate=0.0003", "--run-name", "fno_synth_aiida_lr3e-4", "--mode", "train"],
                    "resubmit_fno", show=SHOW_AIIDA))
print("cloned from", res["cloned_from"], "with", res["overrides_applied"])
wf = describe([res["uuid"]])[res["uuid"]]
assert wf["exit_status"] == 0, wf
TRAINED["fno, lr 3e-4"] = (res["uuid"], wf["outputs"]["model"])

### 3.6 Evaluate on the test collection

`submit eval` scores a model on a collection. It derives the collection in the layout the model trained on, so the
steady derivation is not repeated here. `--full-frames` scores every stored frame (without it an automatic evaluation
subsamples to about 25 frames per case; the steady layout has one frame per case anyway).

The metrics land in the `evaluation_metadata` attribute of a `PinoModelEvaluationData` node. `lpno_repro.metrics`
reads them the way the paper's tables are made: `metric_units.kelvin["T.substrate_abs_err_mean"]` is the T MAE in
kelvin, and so on (the conventions are listed at the top of `lpno_repro.py`).

In [ ]:
EVALS = {}
for name, (_, model) in TRAINED.items():
    res = last_json(run([AIIDA_PINO, "--json", "--foreground", "submit", "eval", "--model", model,
                         "--dataset", COLL["test"], "--python-code", CODE_LABEL, "--full-frames",
                         "--description", f"train_evaluate.ipynb: {name} on the synthetic test split"],
                        f"eval_{name.replace(', ', '_').replace(' ', '')}_aiida", show=SHOW_AIIDA))
    EVALS[name] = describe([res["uuid"]])[res["uuid"]]["outputs"]["evaluation"]

nodes = describe(list(EVALS.values()))
meta = nodes[EVALS["fno"]]["evaluation_metadata"]
print("keys:", sorted(meta))
print("T MAE (K):", meta["metric_units"]["kelvin"]["T.substrate_abs_err_mean"], "| scored cases:", meta["scored_n"])
pd.DataFrame({name: lr.metrics({"evaluation_metadata": nodes[uuid]["evaluation_metadata"]})
              for name, uuid in EVALS.items()}).T.round(3)

Every node above can be traced from the command line: `aiida-pino models`, `aiida-pino model <uuid>`,
`aiida-pino compare-models <a> <b>` (every difference between two models, sorted into confounds and benign changes),
`verdi process list -a`, and `verdi node graph generate <evaluation uuid>` for the provenance graph. With
`AIIDA_PATH` pointing at this work folder's `aiida/` they open the profile made here.

## 4. Choosing between the routes

| | route A, `lpno` | route B, `aiida-pino` |
| --- | --- | --- |
| input | folders of NPZ files | simulation nodes and collections |
| architecture | `--config-name=<config>` | `config_name` in the training profile |
| hyperparameters | Hydra overrides on the command line | structured profile keys, plus `hydra_overrides` |
| steady layout | prepared offline (here written directly; `scripts/build_steady_split.py` for transient runs) | derived from transient runs on demand, stated in the training and derivation profiles |
| result | run folder: weights, `.hydra/`, `fno_eval/model_errors.txt` | model and evaluation nodes with their full provenance |
| re-run | rerun the command | `aiida-pino resubmit` |

## 5. Optional: re-running a published evaluation and training

This section needs the data deposit. It runs only when:

* `LPNO_DEPOSIT` points at the open record (doi:10.5281/zenodo.23325086) and its archives are imported into an AiiDA
  profile (README_AiiDA.md, section 2: `verdi presto`, then `verdi archive import` of `lpno_provenance.aiida` and
  `lpno_provenance_evals.aiida`). `LPNO_AIIDA_PROFILE` names that profile (default: the default profile) and
  `LPNO_AIIDA_PATH` its AiiDA folder (default: the `AIIDA_PATH` of the environment, else `~/.aiida`).
* `LPNO_RESTRICTED` points at the restricted record (doi:10.5281/zenodo.23325088) with its payload tars rejoined
  (`README_restricted.md`) and `payload_manifest.csv`.

It never imports the archives itself (6.8 GB). It does write to that profile: the restored arrays of the simulations it
needs, a code `lpno-tutorial-python@localhost` if missing, and the new evaluation node.

The evaluation is the RIGNO row of the preheat table (`tab:preheat`, evaluation `207db797…`); its UUIDs come from
`data_deposit/provenance_logs/provenance_log.csv`. Following README_AiiDA.md section 7: restore the payloads of what the evaluation scored,
score the same model on the same collection, and compare the new metrics with the deposited ones.

In [ ]:
_dd = CODE / "data_deposit"   # data_deposit/scripts/download.py puts the two records here
DEPOSIT = os.environ.get("LPNO_DEPOSIT") or (str(_dd / "open_record") if (_dd / "open_record" / "lpno_provenance.aiida").exists() else None)
RESTRICTED = os.environ.get("LPNO_RESTRICTED") or (str(_dd / "restricted_record") if any((_dd / "restricted_record").glob("payloads_*.tar")) else None)
REAL_PROFILE = os.environ.get("LPNO_AIIDA_PROFILE", "")
REAL_ENV = {k: v for k, v in ENV.items() if k != "AIIDA_PATH"}      # the user's AiiDA folder, not WORK's
if os.environ.get("LPNO_AIIDA_PATH") or os.environ.get("AIIDA_PATH"):
    REAL_ENV["AIIDA_PATH"] = os.environ.get("LPNO_AIIDA_PATH") or os.environ["AIIDA_PATH"]
PROF = ["--profile", REAL_PROFILE] if REAL_PROFILE else []

log_csv = Path(DEPOSIT or "", "provenance_log.csv")
log_csv = log_csv if DEPOSIT and log_csv.exists() else CODE / "data_deposit" / "provenance_logs" / "provenance_log.csv"
log = pd.read_csv(log_csv)
row = log[(log["table"] == "tab:preheat") & log["evaluation_uuid"].str.startswith("207db797")].iloc[0]
EVAL_UUID, MODEL_UUID = row.evaluation_uuid, row.model_uuid
SCORED_UUID, TRAIN_WF_UUID = row.scored_dataset_uuid, row.training_workflow_uuid


def real_data_missing():
    if not (DEPOSIT and RESTRICTED):
        return "LPNO_DEPOSIT and LPNO_RESTRICTED are not both set"
    if not (Path(RESTRICTED) / "payload_manifest.csv").exists():
        return f"no payload_manifest.csv in {RESTRICTED}"
    found = subprocess.run([VERDI, *(["-p", REAL_PROFILE] if REAL_PROFILE else []), "node", "show", EVAL_UUID],
                           env=REAL_ENV, capture_output=True)
    if found.returncode:
        return f"evaluation {EVAL_UUID} is not in the profile: import the open archives first"
    return None


why = real_data_missing()
REAL = why is None
print(row[["evaluation", "model", "substrate_mae_K", "substrate_rmse_K", "beambox_mae_K", "meltpool_iou", "scored_n"]])
print("\nsection 5 runs" if REAL else f"\nsection 5 skipped: {why}")

In [ ]:
if REAL:
    # a dry run first: how many simulations, how many bytes, which tars
    run([AIIDA_PINO, *PROF, "restore-payloads", "--deposit", RESTRICTED, "--for", EVAL_UUID], "restore_dry_run",
        env=REAL_ENV, tail=20)
    run([AIIDA_PINO, *PROF, "restore-payloads", "--deposit", RESTRICTED, "--for", EVAL_UUID, "--yes"], "restore",
        env=REAL_ENV, tail=10)

    REAL_CODE = "lpno-tutorial-python@localhost"
    exists = subprocess.run([VERDI, *(["-p", REAL_PROFILE] if REAL_PROFILE else []), "code", "show", REAL_CODE],
                            env=REAL_ENV, capture_output=True).returncode == 0
    if not exists:
        run([VERDI, *(["-p", REAL_PROFILE] if REAL_PROFILE else []), "code", "create", "core.code.installed",
             "--non-interactive", "--label", REAL_CODE.split("@")[0], "--computer", "localhost",
             "--filepath-executable", sys.executable, "--prepend-text", "\n".join(prepend)],
            "real_code", env=REAL_ENV)

In [ ]:
if REAL:
    res = last_json(run([AIIDA_PINO, *PROF, "--json", "--foreground", "submit", "eval", "--model", MODEL_UUID,
                         "--dataset", SCORED_UUID, "--python-code", REAL_CODE, "--full-frames",
                         "--description", "re-run of evaluation 207db797 from train_evaluate.ipynb"],
                        "real_eval", env=REAL_ENV, show=SHOW_AIIDA))
    new_eval = describe([res["uuid"]], REAL_PROFILE, REAL_ENV)[res["uuid"]]["outputs"]["evaluation"]
    both = describe([EVAL_UUID, new_eval], REAL_PROFILE, REAL_ENV)
    display(pd.DataFrame({"deposited": lr.metrics(both[EVAL_UUID]), "re-run": lr.metrics(both[new_eval])}))

**Re-running a training** clones the deposited training workflow with `resubmit` (README_AiiDA.md, section 7). A full
32,000-step RIGNO training needs a GPU (`--torch cu126` or `cu128` at install; "Running on a remote GPU computer" in
the aiida-pino README to train on another machine). The cell below runs only with `LPNO_RUN_RESUBMIT=1`, for
`LPNO_RESUBMIT_STEPS` steps (default 20), as a check that the recorded inputs replay: it restores the training and
validation payloads, points the clone at this machine's code (`code_label`) and shortens it. The adapter shortens the
recording and weights-saving intervals with it, so the short run keeps its final weights.

In [ ]:
if REAL and os.environ.get("LPNO_RUN_RESUBMIT") == "1":
    steps = int(os.environ.get("LPNO_RESUBMIT_STEPS", "20"))
    run([AIIDA_PINO, *PROF, "restore-payloads", "--deposit", RESTRICTED, "--for", TRAIN_WF_UUID, "--yes"],
        "restore_training", env=REAL_ENV, tail=10)
    recorded = describe([TRAIN_WF_UUID], REAL_PROFILE, REAL_ENV)[TRAIN_WF_UUID]["training_profile"]
    overrides = {"code_label": REAL_CODE, "max_steps": steps}
    if not recorded.get("gpu_vram_mb"):
        overrides["gpu_vram_mb"] = 16000              # declared, so no measuring run first
    (WORK / "resubmit_overrides.json").write_text(json.dumps(overrides, indent=1))
    res = last_json(run([AIIDA_PINO, *PROF, "--json", "--foreground", "resubmit", TRAIN_WF_UUID,
                         "--override", f"@{WORK / 'resubmit_overrides.json'}",
                         "--run-name", "rigno_preheat107_resubmit_check", "--mode", "train"],
                        "real_resubmit", env=REAL_ENV, show=SHOW_AIIDA))
    print("cloned from", res["cloned_from"], "| overrides:", sorted(res["overrides_applied"]))
else:
    print("training re-run not requested (LPNO_RUN_RESUBMIT=1 with the data of section 5)")

## Where to go next

* `lpno/README.md`: the main Hydra options ("Main options"), the RIGNO and GAOT options ("Architectures"), the metric
  files ("Metrics") and the environment variables of the lazy loader ("Environment variables").
* `aiida-pino/README.md`: the CLI, profiles, steady runs, remote GPU computers, and the web console.
* `data_deposit/record_docs/README_AiiDA.md`: tracing a printed number to its nodes, and re-running it from the deposit.
* `reproducibility/rerun/README.md`: replaying a published training or evaluation without AiiDA, on one GPU or a Slurm cluster.